# Catastrophe Modeling Capstone: 1994 Northridge Earthquake
## Bridge Damage Assessment Using Satellite NDVI & Ground Motion Data

This notebook runs the complete catastrophe modeling pipeline:

| Phase | Module | Description |
|-------|--------|-------------|
| **0** | `modules/ndvi_download` | Download NDVI composites from Google Earth Engine |
| **0b** | `modules/change_detection` | Extract per-bridge NDVI statistics (zonal stats) |
| **0c** | `modules/visualization` | Visualize rasters, scatter plots, spatial maps |
| **1** | `catastrophe_model/damage_classification` | Classify bridges into DS0-DS4 |
| **2** | `catastrophe_model/proxy_fragility` | Derive empirical fragility curves |
| **3** | `catastrophe_model/economic_disruption` | Compute Traffic Disruption Index |
| **4** | `catastrophe_model/prioritization_map` | Emergency priority mapping |

---
## Setup & Configuration

In [ ]:
import sys
import os
import warnings
import pandas as pd
import geopandas as gpd
import numpy as np

warnings.filterwarnings("ignore")

# Add project root to path so modules can be imported
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, PROJECT_ROOT)

# ── Path Configuration ──────────────────────────────────────────────────
# Adjust BASE_DATA_PATH if your DATA folder is in a different location
BASE_DATA_PATH = os.path.join(os.path.dirname(PROJECT_ROOT), "DATA")
CHANGE_DET_DATA = os.path.join(os.path.dirname(PROJECT_ROOT), "change_detection", "datasets")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Key data file paths
BRIDGE_SHP = os.path.join(BASE_DATA_PATH, "Processed_raster_pga", "pga_nbi_bridge.shp")
PRE_NDVI_TIF = os.path.join(CHANGE_DET_DATA, "Pre_Event_NDVI.tif")
POST_NDVI_TIF = os.path.join(CHANGE_DET_DATA, "Post_Event_NDVI.tif")
NDVI_CHANGE_TIF = os.path.join(CHANGE_DET_DATA, "NDVI_Change.tif")
RESULTS_CSV = os.path.join(CHANGE_DET_DATA, "pga_bridge_ndvi_200m.csv")
RESULTS_SHP = os.path.join(CHANGE_DET_DATA, "pga_bridge_ndvi_200m.shp")

print(f"Project root:  {PROJECT_ROOT}")
print(f"Data path:     {BASE_DATA_PATH}")
print(f"Figures dir:   {FIGURES_DIR}")
print(f"Results CSV:   {os.path.exists(RESULTS_CSV)}")

---
## Phase 0: NDVI Download (Optional - Run if data not already downloaded)

Skip this section if you already have the NDVI TIFFs in `change_detection/datasets/`.

**AOI Options** — pick the one that matches your ShakeMap coverage:

| Method | Use when... |
|--------|-------------|
| `get_aoi_county("Los Angeles")` | ShakeMap covers ~1 county |
| `get_aoi_multi_county(["Los Angeles", "Ventura", ...])` | ShakeMap covers several counties |
| `get_aoi_state("California")` | ShakeMap covers the whole state |
| `get_aoi_bbox(west, south, east, north)` | You want a custom bounding box |
| `get_aoi_from_raster("path/to/pga_mean.flt")` | **Best** — exactly match your ShakeMap extent |

In [ ]:
# ── Uncomment and run ONLY if you need to re-download NDVI from GEE ──
# import ee
# from modules.ndvi_download import (
#     download_ndvi_composites,
#     get_aoi_county, get_aoi_state, get_aoi_multi_county,
#     get_aoi_bbox, get_aoi_from_raster,
# )
#
# # Step 1: Authenticate & Initialize FIRST (required before any ee.Geometry call)
# ee.Authenticate()
# ee.Initialize(project="ee-mat724")
#
# # Step 2: Pick ONE AOI option ───────────────────────────────────────────
#
# # Option 1: Single county
# # aoi = get_aoi_county("Los Angeles")
#
# # Option 2: Whole state of California
# # aoi = get_aoi_state("California")
#
# # Option 3: Multiple counties around the epicenter
# # aoi = get_aoi_multi_county(["Los Angeles", "Ventura", "Orange",
# #                              "San Bernardino", "Kern", "Santa Barbara"])
#
# # Option 4: Custom bounding box
# # aoi = get_aoi_bbox(west=-121.5, south=33.0, east=-116.0, north=36.5)
#
# # Option 5: BEST — match your ShakeMap raster extent exactly
# # PGA_RASTER = os.path.join(BASE_DATA_PATH, "raster", "pga_mean.flt")
# # aoi = get_aoi_from_raster(PGA_RASTER)
#
# # Step 3: Download ─────────────────────────────────────────────────────
# paths = download_ndvi_composites(
#     aoi=aoi,
#     disaster_date="1994-01-17",
#     output_dir=CHANGE_DET_DATA,
#     window_days=60,
#     scale=30,
# )
# print("Downloaded:", paths)

## Phase 0b: Change Detection - Extract Bridge NDVI (Optional)

Skip if `pga_bridge_ndvi_200m.csv` already exists.

In [ ]:
# ── Uncomment and run ONLY if you need to re-extract bridge NDVI ──
# from modules.change_detection import extract_bridge_ndvi
#
# bridges = extract_bridge_ndvi(
#     bridge_shp_path=BRIDGE_SHP,
#     pre_ndvi_path=PRE_NDVI_TIF,
#     post_ndvi_path=POST_NDVI_TIF,
#     buffer_m=200,
#     output_csv=RESULTS_CSV,
#     output_shp=RESULTS_SHP,
# )

---
## Load Pre-Computed Results

Load the bridge-level dataset with PGA, NDVI, and traffic data.

In [ ]:
# Load tabular results
df = pd.read_csv(RESULTS_CSV).dropna(subset=["pga", "ndvi_chan"])

# Load spatial results
gdf = gpd.read_file(RESULTS_SHP).dropna(subset=["ndvi_chan"])

print(f"Loaded {len(df):,} bridges with valid PGA and NDVI data")
print(f"\nKey columns: {list(df.columns)}")
print(f"\nSample:")
df[["lat", "long", "pga", "ndvi_chan", "avg_daily_"]].head()

## Phase 0c: Visualization - NDVI Rasters & Scatter Analysis

In [ ]:
from modules.visualization import plot_ndvi_rasters, plot_ndvi_change_only, plot_pga_vs_ndvi, plot_bridge_ndvi_map

# Three-panel NDVI map
plot_ndvi_rasters(
    PRE_NDVI_TIF, POST_NDVI_TIF, NDVI_CHANGE_TIF,
    save_path=os.path.join(FIGURES_DIR, "NDVI_Raster_Maps.png")
)

In [ ]:
# NDVI change only
plot_ndvi_change_only(
    NDVI_CHANGE_TIF,
    save_path=os.path.join(FIGURES_DIR, "NDVI_Change_Only.png")
)

In [ ]:
# PGA vs NDVI scatter
plot_pga_vs_ndvi(df, save_path=os.path.join(FIGURES_DIR, "PGA_vs_NDVI.png"))

In [ ]:
# Spatial bridge NDVI map
plot_bridge_ndvi_map(gdf, save_path=os.path.join(FIGURES_DIR, "Bridge_NDVI_Map.png"))

---
## Phase 1: Damage State Classification

In [ ]:
from catastrophe_model.damage_classification import (
    classify_damage_states, plot_damage_distribution, print_damage_summary
)

# Classify all bridges
df = classify_damage_states(df)

# Summary
print_damage_summary(df)

In [ ]:
# Visualization
plot_damage_distribution(df, save_path=os.path.join(FIGURES_DIR, "Phase1_DamageClassification.png"))

---
## Phase 2: Empirical Fragility Curves

In [ ]:
from catastrophe_model.proxy_fragility import (
    compute_fragility_curves, plot_fragility_curves, print_fragility_summary
)

# Compute empirical + fitted fragility
empirical, fitted_params = compute_fragility_curves(df)

# Summary
print_fragility_summary(fitted_params)

In [ ]:
# Visualization
plot_fragility_curves(
    empirical, fitted_params,
    save_path=os.path.join(FIGURES_DIR, "Phase2_FragilityCurves.png")
)

---
## Phase 3: Economic Disruption (Traffic Disruption Index)

In [ ]:
from catastrophe_model.economic_disruption import compute_tdi, summarize_tdi, plot_tdi

# Compute TDI
df = compute_tdi(df)

# Summary
summarize_tdi(df)

In [ ]:
# Visualization
plot_tdi(df, save_path=os.path.join(FIGURES_DIR, "Phase3_EconomicDisruption.png"))

---
## Phase 4: Emergency Prioritization Map

In [ ]:
from catastrophe_model.prioritization_map import create_priority_map, get_top_priority_bridges

# Merge damage/TDI results back onto the spatial dataset
gdf_merged = gdf.merge(
    df[["lat", "long", "damage_state", "TDI_weighted", "severity_weight"]],
    on=["lat", "long"],
    how="left",
)
gdf_merged["damage_state"] = gdf_merged["damage_state"].fillna("DS0 \u2013 No Damage")
gdf_merged["TDI_weighted"] = gdf_merged["TDI_weighted"].fillna(0)

# Priority map
create_priority_map(
    gdf_merged,
    save_path=os.path.join(FIGURES_DIR, "Phase4_PriorityMap.png")
)

In [ ]:
# Top 20 priority bridges
top20 = get_top_priority_bridges(df, n=20)

---
## Export Final Results

In [ ]:
# Save final enriched dataset
output_csv = os.path.join(DATA_DIR, "final_bridge_analysis.csv")
df.to_csv(output_csv, index=False)
print(f"Final results saved: {output_csv}")
print(f"Total bridges: {len(df):,}")
print(f"Columns: {list(df.columns)}")

# List all generated figures
print(f"\nFigures saved in: {FIGURES_DIR}")
for f in sorted(os.listdir(FIGURES_DIR)):
    print(f"  - {f}")

---
## Summary

This pipeline implements a complete catastrophe model:

1. **Hazard**: PGA from USGS ShakeMap rasters
2. **Exposure**: 19,868 bridges from National Bridge Inventory with ADT
3. **Vulnerability**: NDVI-based damage classification + empirical fragility curves
4. **Loss**: Traffic Disruption Index as economic disruption proxy

The output is an actionable emergency priority map that surfaces bridges requiring immediate inspection based on both structural damage severity and traffic criticality.